In [ ]:
import pandas as pd
import json
import os
from tqdm.notebook import tqdm
from pydantic import RootModel
import argparse
from typing import List
import ollama
import time
import pandas as pd
import csv
import re
from sklearn.metrics import classification_report


In [ ]:
# HELPER FUNCTIONS

def regex_extract_to_dataframe(strings):
    # Initialize empty lists to store extracted values
    article_ids = []
    about_immigration_values = []
    relevant_to_nl_values = []

    # Define regex pattern for article_id and about_immigration with optional double quotes
    article_id_pattern = r'"article_id"\s*:\s*"?(\d+)"?'
    about_immigration_pattern = r'"main_topic_immigration"\s*:\s*"?(\d)"?'
    relevant_to_nl_pattern = r'"relevant_to_nl"\s*:\s*"?(\d)"?'

    # Iterate through each string
    for string_data in strings:
        # replace \ with nothing
        string_data = string_data.replace("\\", "")
        string_data = string_data.lower()
        
        # Use regex to find matches for article_id
        article_id_match = re.search(article_id_pattern, string_data)

        # Use regex to find matches for about_immigration
        about_immigration_match = re.search(about_immigration_pattern, string_data)

        # Use regex to find matches for relevant_to_nl
        relevant_to_nl_match = re.search(relevant_to_nl_pattern, string_data)

        # Extract values from the regex matches
        article_id = int(article_id_match.group(1)) if article_id_match else None
        about_immigration = int(about_immigration_match.group(1)) if about_immigration_match else None
        relevant_to_nl = int(relevant_to_nl_match.group(1)) if relevant_to_nl_match else None

        # Append values to the respective lists
        article_ids.append(article_id)
        about_immigration_values.append(about_immigration)
        relevant_to_nl_values.append(relevant_to_nl)

    # Create a DataFrame using the extracted values
    df = pd.DataFrame({
        "article_id": article_ids,
        "about_immigration": about_immigration_values,
        "relevant_to_nl": relevant_to_nl_values
    })

    return df

def build_prompt(article_id: str, text: str, system_prompt: str, main_prompt: str):
    """Create prompts in the format expected by Ollama"""
    # Format the prompts with actual values
    # formatted_input = input_prompt.format(article_id=article_id, text=text)
    formatted_main = main_prompt.format(article_id=article_id, text=text)
    
    # Combine user content
    # user_content = formatted_input + formatted_main
    user_content = formatted_main
    
    messages = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user", 
            "content": user_content
        }
    ]
    return messages

In [ ]:
# ANNOTATED DATA FOR PROMPT TESTING
df = pd.read_csv('/home/eklk/data/storage_100gb/election_news/data/nl_annotated_test.csv', encoding='utf-8-sig', sep=';', quoting=csv.QUOTE_NONNUMERIC)
print(df.shape)
df['article_id'] = df['article_id'].astype(int)
# Rename Text to text
df.rename(columns={'Text': 'text'}, inplace=True)
# merge article_title and text for text
df['text'] = df['article_title'] + '\n' + df['text']
df.head()

In [ ]:
system_prompt = """
You are a professional news annotator specializing in identifying whether articles are mainly about immigration and if so, whether the immigration discussion in the article is relevant to the Netherlands. 
"""

In [ ]:
main_prompt = """
An article is considered mainly about immigration if the majority of its coverage focuses on immigration or immigrant groups (including immigrants, asylum seekers, refugees, resettlers, expats, international students, foreign workers, seasonal workers). Consider the complete article text. If immigration-related content comprises at least half of the article, code it as 1. 

If immigration-related content comprises approximately the half of discussion, consider the title and article framing to decide whether the discussion around immigration is sufficient to be considered as the main topic of the article. If immigration is one of two central, equally developed themes, it still counts as immigration coverage. The key test is: Remove the immigration context – does the core story still exist? If yes, classify as 0. 

Do **not** count brief mentions of immigration as main topic if the article is mainly about other issues (e.g., anti-Muslim sentiment, religious intolerance, racism, or ethnic discrimination, crime stories where the perpetrator or the victim is an immigrant with no further discussion on immigration or immigrant groups), discussions around emigration (the departure of citizens/professionals from the country of newspaper abroad) or migration within country and discussions around short-term travel or visas (e.g., Schengen tourist visas, 90-day stays, holiday travel) unless the coverage of such issues is directly linked and discussed in relation to immigration or immigrant groups in the larger portion of the article text.

Include all forms of immigration discussions such as:
- Immigration and immigrant groups (asylum seekers, refugees, expats, international students, foreign workers, seasonal workers) and their experiences with migration, immigration processes and integration into host countries, 
- Private stories and lived experiences of immigrants and their families, including personal challenges, challenges during migration, adaptation, integration, work and community life in the host country,
- Immigration related policy and societal topics such as migration flows, border controls, deportations, labor migration, visas/residence permits/naturalization/resettlement, housing and accommodation of immigrants, and social integration programs,
- Illegal immigration including unauthorized border crossings, people smuggling networks and organized criminal activities related to immigration, misconduct or criminal cases involving immigration processes, 
- The economic/financial and social impacts of immigration on both origin and destination countries, 
- Political discussions, legislations and campaign rhetoric related to immigration policies and/or immigrant groups,
- Public opinion about immigration and immigrants, 
- Bilateral agreements, international treaties, and EU regulations (if applicable) governing immigration and mobility.

If the article is mainly about immigration, also decide whether the discussion is relevant to the Netherlands context. When coding this variable, consider only the parts of the article about immigration as the context unit and ignore the rest of the article. 

Code this variable as 1 if the article discusses:
- General discussions on immigration, migration, immigrant groups without specific reference to any country,
- Immigration/asylum/integration directly affecting the Netherlands,
- Immigration policies or regulations that apply to the Netherlands,
- Immigration flows or situations in neighboring or key origin/transit countries that impact the immigration context/discussions in the Netherlands,
- Immigration discussions in other countries such as key origin/transit countries that are linked/presented in comparison/relation to the Netherlands, 
- Bilateral or multilateral agreements related to immigration involving the Netherlands,
- Immigration-related events or developments in other countries that have direct implications for immigration policy or public debate of the Netherlands.

Code this variable as 0 if the article discusses:
- Immigration issues in distant countries to the Netherlands,
- Immigration issues in countries with no clear connection to the Netherlands, 
- Immigration stories or policies that do not impact or relate to the context of the Netherlands. 

Use the following JSON format for your output. Your output should only contain the JSON object without any additional text or explanation.

```json
{{
  "article_id": "{article_id}",
  "main_topic_immigration": 1 or 0,
  "relevant_to_nl": 1 or 0
}}
```

Article to annotate:

Article ID: {article_id}
Text: {text}

Task: Using these instructions, classify whether this article is mainly about immigration. Then, if main_topic_immigration = 1, also classify whether the immigration discussion in the article is relevant to the Netherlands.
- Provide a value (1 or 0) for "relevant_to_nl" **only if main_topic_immigration = 1**, otherwise set "relevant_to_nl" to 0.
- Output your response in the provided JSON format, without any additional text or explanation.
"""

In [ ]:
# Process dataset
unique_articles = df[['article_id', 'text', 'main_topic']].drop_duplicates(subset=['article_id']).reset_index(drop=True)

In [ ]:
model_name = "phi4-mini:3.8b" 
ollama.pull(model_name)

In [ ]:
%time
results = []

for idx, row in unique_articles.iterrows():
    article_id = row['article_id']
    text = row['text']

    messages = build_prompt(article_id, text, system_prompt, main_prompt)
    
    response = ollama.chat(
        model=model_name,  
        messages=messages,
        options={'temperature': 0.0, 'seed': 0}
    )

    response_content = response.message.content
    print(response_content)

    results.append(response_content)

In [ ]:
coded_df = regex_extract_to_dataframe(results)
coded_df.head()

In [ ]:
coded_df.isnull().sum()
# drop if article_id is null
coded_df = coded_df[coded_df['article_id'].notnull()]

coded_df['article_id'] = coded_df['article_id'].astype(int)
coded_df['about_immigration'] = coded_df['about_immigration'].astype(int)   
coded_df['relevant_to_nl'] = coded_df['relevant_to_nl'].astype(int)   

coded_df.isnull().sum()


In [ ]:
# in coded_df if about_immigration is 0, set relevant_to_nl to None
coded_df.loc[coded_df['about_immigration'] == 0, 'relevant_to_nl'] = None

In [ ]:
# left join the coded_df with the original df
df_merged = coded_df.merge(df, on='article_id', how='left')
print(df_merged.shape)

In [ ]:
pd.crosstab(df_merged['about_immigration'], df_merged['main_topic'])

In [ ]:
print(classification_report(df_merged['about_immigration'], df_merged['main_topic']))

In [ ]:
df_merged_immigration = df_merged[(df_merged['about_immigration'] == 1) & (df_merged['main_topic'] == 1)]
df_merged_immigration.shape

In [ ]:
pd.crosstab(df_merged_immigration['relevance'], df_merged_immigration['relevant_to_nl'], dropna=False)

In [ ]:
print(classification_report(df_merged_immigration['relevance'], df_merged_immigration['relevant_to_nl']))

In [ ]:
df_merged['main_topic'] = df_merged['main_topic'].astype(int)
df_merged['about_immigration'] = df_merged['about_immigration'].astype(int)
df_merged['relevant_to_nl'] = df_merged['relevant_to_nl'].astype('Int64')

In [ ]:
# write df_merged to csv
df_merged.to_csv('nl_test_phi4mini_mainprompt.csv', index=False, sep=';', quoting=csv.QUOTE_NONNUMERIC)